# Silver · chamados

**Origem:** `vcredit_bronze.chamados` → **Destino:** `vcredit_silver.chamados`

**Grão:** uma linha por chamado. **Depende de:** bronze

Tabela central dos atendimentos: cliente, motivo, canal, resolução, tempos registrados e atendente de cada chamado.

| Regra | Tratamento |
|---|---|
| `id_chamado` | Convertido para número; IDs repetidos são removidos, mantendo um (as repetições da bronze são cópias idênticas) |
| `id_cliente` | Espaços sobrando removidos; mantido como texto (preserva zeros à esquerda) |
| `motivo` | Sem acento, minúsculo e sem espaços sobrando (mesmo padrão de `silver.motivos`) |
| `canal` | Variações ("U.R.A", "Chat bot", "Atend. Inicial"...) mapeadas para ura, chatbot, atendimento inicial e atendimento especializado |
| `resolvido` | "Sim", "S", "1" viram `sim`; "Não", "N", "0", "Nao resolvido" viram `nao` |
| `tempo_espera` e `tempo_atendimento` | Convertidos para número; "NULL" e valores menores ou iguais a zero viram nulo |
| `id_atendente` | Convertido para número; IDs 0, 99 e 999 marcados em `flag_atendente_placeholder` |

**Decisões:** os IDs de atendente 0, 99 e 999 não existem no cadastro; são preservados e marcados, não apagados, para manter a evidência da inconsistência. As colunas de horário ficam vazias nesta tabela: os horários vêm de `silver.chamados_hora` e são cruzados na gold. Os tempos registrados não passam de ~999 s; os tempos reais são calculados na gold a partir dos horários.

Pipeline idempotente (lê da bronze e sobrescreve).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "rocket_lab"

bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

In [0]:
# Importação da tabela chamado da camada bronze
bronze_chamado = spark.table(f"{bronze_schema}.chamados")

# Transformação e padronização da coluna id_camado
silver_chamado = (
    bronze_chamado
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .dropDuplicates(["id_chamado"])
)

# Transformação e padronização da coluna id_cliente
silver_chamado = (
    silver_chamado
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
)

COM_ACENTO = "áàâãäéèêëíìîïóòôõöúùûüçñÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

# Transformação e padronização da coluna motivo
silver_chamado = (
    silver_chamado.
    withColumn("motivo", F.lower(
        F.regexp_replace(
            F.trim(F.translate(F.regexp_replace(F.col("motivo"), "[\\u0300-\\u036f]", ""), COM_ACENTO, SEM_ACENTO)), r"\s+", " "
        )))
)

# Transformação e padronização da coluna canal
silver_chamado = (
    silver_chamado
    .withColumn("canal", F.when(F.col("canal").isin(["Ura", "U.R.A", "URA"]), "ura")
                          .when(F.col("canal").isin(["CHATBOT", "ChatBot", "Chat bot", "Chatbot", "chat bot", "Chat Bot"]), "chatbot")
                          .when(F.col("canal").isin(["ATENDIMENTO INICIAL", "Atend. Inicial", "Atendimento_Inicial", "Atendimento Inicial"]), "atendimento inicial")
                          .when(F.col("canal").isin(["ATENDIMENTO ESPECIALIZADO", "Atend. Especializado", "Atendimento_Especializado", "Atendimento Especializado"]), "atendimento especializado")
                          .otherwise(F.col("canal")))
)

# Transformação e padronização da coluna resolvido
silver_chamado = (
    silver_chamado
    .withColumn("resolvido", F.when(F.col("resolvido").isin(["Sim", "S", "1"]), "sim")
                          .when(F.col("resolvido").isin(["Não", "N", "0", "Nao resolvido"]), "nao")
                          .otherwise(F.col("resolvido")))
)

# Transformação e padronização da coluna hora_inicio_atendimento
silver_chamado = (
    silver_chamado
    .withColumn("hora_inicio_atendimento", F.lower(F.trim(F.col("hora_inicio_atendimento"))))
)

# Transformação e padronização da coluna tempo_espera
silver_chamado = (
    silver_chamado
    .withColumn("tempo_espera", F.expr("try_cast(nullif(upper(trim(tempo_espera)), 'NULL') as int)"))
    .withColumn("tempo_espera", F.when(F.col("tempo_espera") <= 0, None).otherwise(F.col("tempo_espera")))
)

# Transformação e padronização da coluna tempo_atendimento
silver_chamado = (
    silver_chamado
    .withColumn("tempo_atendimento", F.expr("try_cast(nullif(upper(trim(tempo_atendimento)), 'NULL') as int)"))
    .withColumn("tempo_atendimento", F.when(F.col("tempo_atendimento") <= 0, None).otherwise(F.col("tempo_atendimento")))
)

# Transformação e padronização da coluna id_atendente
# Adição de uma nova coluna flag_atendente_placeholder para verificar, na camada gold, os IDs de atendentes com valores 0, 99 e 999
silver_chamado = (
    silver_chamado
    .withColumn("id_atendente", F.expr("try_cast(trim(id_atendente) as int)"))
    .withColumn("flag_atendente_placeholder", F.col("id_atendente").isin([0, 99, 999]))
)

 # Idempotente
(
    silver_chamado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.chamados")
)

display(silver_chamado.orderBy("id_chamado").limit(10))